In [148]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pandas as pd
import io

# Step 1: Set credentials
SERVICE_ACCOUNT_FILE = 'C:\\Users\\naing\\MultiVacSim\\spring25research.json'
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)

# Step 2: Build service
drive_service = build('drive', 'v3', credentials=creds)

# Step 3: Function to load CSV from Drive
def load_csv_from_drive(file_id):
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    fh.seek(0)
    return pd.read_csv(fh)

# # Example: COVID hospitalization file
# df = load_csv_from_drive("1euq3iccfFdFX9ipkuX5j-r3z3PtonVBk")
# print(df.head())


In [ ]:
import gym 
from gym import spaces
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import sys
import os
# ----------------------
# Data Loader
# ----------------------
def normalize(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-8)

def clean_numeric_column(series):
    return pd.to_numeric(series.astype(str).str.replace('%', '', regex=False).str.replace('>', '', regex=False), errors='coerce')

# HHS region mapper
def map_state_to_hhs_region(state):
    # Normalize input
    state = state.strip()
    
    # Mapping from full state names to abbreviations
    state_to_abbrev = {
        "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
        "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "District of Columbia": "DC",
        "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL",
        "Indiana": "IN", "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA",
        "Maine": "ME", "Maryland": "MD", "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
        "Mississippi": "MS", "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
        "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY", "NYC": "NYC",
        "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK", "Oregon": "OR",
        "Pennsylvania": "PA", "Puerto Rico": "PR", "Rhode Island": "RI", "South Carolina": "SC",
        "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
        "Virginia": "VA", "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY"
    }

    # Convert full name to abbreviation if needed
    if len(state) > 2:
        state = state_to_abbrev.get(state.title(), "").upper()
    else:
        state = state.upper()

    hhs_map = {
        1: {"CT", "ME", "MA", "NH", "RI", "VT"},
        2: {"NJ", "NY", "NYC", "PR"},
        3: {"DE", "DC", "MD", "PA", "VA", "WV"},
        4: {"AL", "FL", "GA", "KY", "MS", "NC", "SC", "TN"},
        5: {"IL", "IN", "MI", "MN", "OH", "WI"},
        6: {"AR", "LA", "NM", "OK", "TX"},
        7: {"IA", "KS", "MO", "NE"},
        8: {"CO", "MT", "ND", "SD", "UT", "WY"},
        9: {"AZ", "CA", "HI", "NV"},
        10: {"AK", "ID", "OR", "WA"}
    }

    for region, states in hhs_map.items():
        if state in states:
            return f"Region {region}"
    return "Other"

# def load_data(disease_type=None, data_level=None, geo_level=None):
def load_data(disease_type=None, data_level=None, geo_level=None, start_date=None, end_date=None):
    disease_type = disease_type.upper() if disease_type else "COVID"
    data_level = data_level.lower() if data_level else "normalized"
    geo_level = geo_level.lower() if geo_level else "national"
    # Validate values
    if disease_type not in {"COVID", "FLU"}:
        print(f"❌ Invalid disease_type: {disease_type}")
        return None
    if data_level not in {"raw", "normalized"}:
        print(f"❌ Invalid level: {data_level}")
        return None
    if geo_level not in {"national", "regional", "state"}:
        print(f"❌ Invalid geo_level: {geo_level}")
        return None

    # File IDs for each dataset by geo level
    file_ids = {
        "COVID": {
            "national": {
                "hospital": "1euq3iccfFdFX9ipkuX5j-r3z3PtonVBk",
                "death": "1ML_bLFZvl18KjRk0X7u1bZ3j44qWxjH4",
                # "vaccine": "1v47Sqi5dcOERvz3wHWTBGAjXLas3VmWq"
                "vaccine": "1AnqrZLF156SQ5pJkwq6Yj68b0LTZP6hc",
            },
            "regional": {
                "hospital": "1euq3iccfFdFX9ipkuX5j-r3z3PtonVBk",
                "death": "10TkIhHkJW04sVpE1uJXWq8jXtSdldBu3",
                "vaccine": "1v47Sqi5dcOERvz3wHWTBGAjXLas3VmWq",
            },
            "state": {
                "hospital": "1euq3iccfFdFX9ipkuX5j-r3z3PtonVBk",
                "death": "1ML_bLFZvl18KjRk0X7u1bZ3j44qWxjH4",
                "vaccine": "1v47Sqi5dcOERvz3wHWTBGAjXLas3VmWq"
            },
        },
        "FLU": {
            "national": {
                "death": "1xlwDm2jmrn5NwV45PWMIie_--TBdV8bN",
                "cumulative": "1cmxUyiXb-ISzel-8OAh6wiXsZ87OBhhU",
                "seasonal": "1PiJ5dXEirrsFELXtNE50XVkPOOJq-cqb"
            },
            "regional": {
                "death": "150cLlW4BaGyYKEFOh1mdEwIriCOu2VVk",
                "cumulative": "1cmxUyiXb-ISzel-8OAh6wiXsZ87OBhhU",
                "seasonal": "1PiJ5dXEirrsFELXtNE50XVkPOOJq-cqb",        
            },
            "state": {
                "death": "1l8Pyny1_mfXYVcBF55QYRu_oYBPDU6Gh",
                "cumulative": "1cmxUyiXb-ISzel-8OAh6wiXsZ87OBhhU",
                "seasonal": "1PiJ5dXEirrsFELXtNE50XVkPOOJq-cqb",        
            },
        }
    }
    if disease_type == "COVID":
        ids = file_ids.get(disease_type.upper(), {}).get(geo_level, {})
        if not ids:
            print(f"❌ No files found for {disease_type} at {geo_level} level.")
            return None
    
        try:
            print("📥 Loading files from Google Drive...")
            hosp = load_csv_from_drive(ids["hospital"])
            deaths = load_csv_from_drive(ids["death"])
            vaccine = load_csv_from_drive(ids["vaccine"])

            print(f"[DEBUG] hosp shape: {hosp.shape}")
            print(f"[DEBUG] deaths shape: {deaths.shape}")
            print(f"[DEBUG] vaccine shape: {vaccine.shape}")
            print(f"[DEBUG] hosp columns: {hosp.columns.tolist()}")
            print(f"[DEBUG] deaths columns: {deaths.columns.tolist()}")
            print(f"[DEBUG] vaccine columns: {vaccine.columns.tolist()}")
            # Time filtering
            if start_date and end_date and 'date' in hosp.columns:
                hosp['date'] = pd.to_datetime(hosp['date'], errors='coerce')
                hosp = hosp[(hosp['date'] >= start_date) & (hosp['date'] <= end_date)]
            if start_date and end_date and 'date' in deaths.columns:
                deaths['date'] = pd.to_datetime(deaths['date'], errors='coerce')
                deaths = deaths[(deaths['date'] >= start_date) & (deaths['date'] <= end_date)]

            hosp_col = hosp.columns[-1]
            # death_col = deaths.columns[-1]
            death_col = "COVID-19 Deaths"
            print(f"[DEBUG] Selected hospitalization column: {hosp_col}")
            print(f"[DEBUG] Selected death column: {death_col}")

            # Convert to numeric and drop NaNs
            hosp[hosp_col] = pd.to_numeric(hosp[hosp_col], errors='coerce')
            deaths[death_col] = pd.to_numeric(deaths[death_col], errors='coerce')
            hosp = hosp.dropna(subset=[hosp_col])
            deaths = deaths.dropna(subset=[death_col])

            print(f"[DEBUG] hosp rows after dropping NaNs: {len(hosp)}")
            print(f"[DEBUG] deaths rows after dropping NaNs: {len(deaths)}")
            print("[DEBUG] hosp columns:", hosp.columns.tolist())
            print("[DEBUG] deaths columns:", deaths.columns.tolist())

            # ✅ Add HHS region mapping
            # hosp["region"] = hosp["State"].apply(map_state_to_hhs_region)
            # deaths["region"] = deaths["HHS Region"].apply(map_state_to_hhs_region)
            # # Group by region
            # hosp_grouped = hosp.groupby("region")[hosp_col].mean().reset_index(name="hospitalization_rate")
            # deaths_grouped = deaths.groupby("region")[death_col].mean().reset_index(name="death_rate")
            # merged = pd.merge(hosp_grouped, deaths_grouped, on="region")

            # Valid state names and abbreviations (uppercase and title case)
            valid_states = {
                "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN", "IA",
                "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
                "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT",
                "VA", "WA", "WV", "WI", "WY",
                "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware", "Florida", "Georgia",
                "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts",
                "Michigan", "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey",
                "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island",
                "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia",
                "Wisconsin", "Wyoming"
            }

            # Clean up and filter first
            hosp["State"] = hosp["State"].str.upper().str.strip()
            hosp = hosp[hosp["State"].isin(valid_states)]
            print(f"[DEBUG] hosp rows after state filtering: {len(hosp)}")

            # Filter deaths to HHS Regions 1–10
            deaths["HHS Region"] = deaths["HHS Region"].astype(str).str.strip()
            deaths = deaths[deaths["HHS Region"].isin([str(i) for i in range(1, 11)])]
            print(f"[DEBUG] deaths rows after region filtering: {len(deaths)}")
            
            # State name → abbreviation mapper
            us_state_to_abbrev = {
                'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
                'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
                'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
                'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
                'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO',
                'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ',
                'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND',
                'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI',
                'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
                'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI',
                'Wyoming': 'WY', 'District of Columbia': 'DC', 'Puerto Rico': 'PR', 'New York City': 'NYC'
            }

            # Normalize to abbreviations before mapping
            hosp["State"] = hosp["State"].map(lambda x: us_state_to_abbrev.get(str(x).strip().title(), ""))

            # Fix region mapping
            hosp["region"] = hosp["State"].apply(map_state_to_hhs_region)
            deaths["region"] = "Region " + deaths["HHS Region"].astype(str)

            # Group and merge
            hosp_grouped = hosp.groupby("region")[hosp_col].mean().reset_index(name="hospitalization_rate")
            deaths_grouped = deaths.groupby("region")[death_col].mean().reset_index(name="death_rate")
            print("[DEBUG] hosp_grouped:", hosp_grouped.head())
            print("[DEBUG] deaths_grouped:", deaths_grouped.head())
            
            merged = pd.merge(hosp_grouped, deaths_grouped, on="region")
            print(f"[DEBUG] merged shape: {merged.shape}")
    
            # ➕ Assign same vaccine stats to all regions
            total_doses = vaccine["Cumulative_Doses"].dropna().astype(float).sum()
            mean_dose = vaccine["Dose"].dropna().astype(float).mean() if "Dose" in vaccine.columns else 0
            total_population = 330_000_000
            vaccine_supply_ratio = total_doses / total_population
            dose_rate = mean_dose / 100_000

            merged["vaccine_supply_ratio"] = vaccine_supply_ratio
            merged["mean_weekly_dose"] = dose_rate

            # Normalize or scale down
            if data_level == "normalized":
                merged["hospitalization_rate"] = normalize(merged["hospitalization_rate"])
                merged["death_rate"] = normalize(merged["death_rate"])
            else:
                merged["hospitalization_rate"] /= 100
                merged["death_rate"] /= 100

            df = merged.rename(columns={"region": "region"})
            print(f"[DEBUG] Final dataframe preview:\n{df.head()}")
            return df[["region", "hospitalization_rate", "death_rate", "vaccine_supply_ratio", "mean_weekly_dose"]].dropna()

        except Exception as e:
            print(f"[ERROR loading COVID data]: {e}")
            return None

    elif disease_type == "FLU":
        base_path = "C:\\Users\\naing\\MultiVacSim\\data\\flu\\processed"

        if geo_level == "national":
            hosp_path = os.path.join(base_path, "NHSN_National.csv")
        elif geo_level == "regional":
            hosp_path = os.path.join(base_path, "NHSN_HHS_Region.csv")
        elif geo_level == "state":
            hosp_path = os.path.join(base_path, "NHSN_State.csv")

        ids = file_ids["FLU"].get(geo_level)
        if not ids:
            print(f"❌ No FLU data available for geo level: {geo_level}")
            return None

        try:
            hosp = pd.read_csv(hosp_path)
            deaths = load_csv_from_drive(ids["death"])
            cumulative = load_csv_from_drive(ids["cumulative"])
            seasonal = load_csv_from_drive(ids["seasonal"])

            # Time filtering
            if start_date and end_date and 'date' in hosp.columns:
                hosp['date'] = pd.to_datetime(hosp['date'], errors='coerce')
                hosp = hosp[(hosp['date'] >= start_date) & (hosp['date'] <= end_date)]
            if start_date and end_date and 'date' in deaths.columns:
                deaths['date'] = pd.to_datetime(deaths['date'], errors='coerce')
                deaths = deaths[(deaths['date'] >= start_date) & (deaths['date'] <= end_date)]

            hosp_col = hosp.columns[-1]
            death_col = deaths.columns[-1]
            hosp[hosp_col] = pd.to_numeric(hosp[hosp_col], errors='coerce')
            deaths[death_col] = pd.to_numeric(deaths[death_col], errors='coerce')
            hosp = hosp.dropna(subset=[hosp_col])
            deaths = deaths.dropna(subset=[death_col])

            cumulative_total = pd.to_numeric(cumulative.iloc[:, -1], errors='coerce').dropna().max() * 1e6  # last cumulative value in millions
            # cumulative_total = cumulative["Cumulative_Flu_Doses"].dropna().astype(float).sum()
            seasonal_avg = pd.to_numeric(seasonal.iloc[:, -1], errors='coerce').dropna().mean() * 1e6        # mean seasonal dose
            # seasonal_avg = mean_dose = seasonal["Dose"].dropna().astype(float).mean() if "Dose" in vaccine.columns else 0
            total_population = 330_000_000
            vaccine_supply_ratio = cumulative_total / total_population
            mean_weekly_dose = seasonal_avg / (52 * 100_000)

            # --- USE THIS: per-region grouping ---
            region_col = "region"  # ✅ Update if needed (e.g., "state", "HHS Region")
            # Convert to numeric and drop NaNs
            hosp[hosp_col] = pd.to_numeric(hosp[hosp_col], errors='coerce')
            deaths[death_col] = pd.to_numeric(deaths[death_col], errors='coerce')
            hosp = hosp.dropna(subset=[hosp_col])
            deaths = deaths.dropna(subset=[death_col])

            # Only keep rows where HHS Region is in [1-10]
            deaths = deaths[deaths["HHS Region"].astype(str).str.strip().isin([str(i) for i in range(1, 11)])]

            # ✅ Add this: HHS region mapping and aggregation
            hosp["region"] = hosp["state"].apply(map_state_to_hhs_region)
            # deaths["region"] = deaths["state"].apply(map_state_to_hhs_region)
            deaths["region"] = "Region " + deaths["HHS Region"].astype(str)

            hosp_grouped = hosp.groupby("region")[hosp_col].mean().reset_index(name="hospitalization_rate")
            deaths_grouped = deaths.groupby("region")[death_col].mean().reset_index(name="death_rate")
            merged = pd.merge(hosp_grouped, deaths_grouped, on="region")

            # ➕ Add vaccine stats to all regions
            merged["vaccine_supply_ratio"] = vaccine_supply_ratio
            merged["mean_weekly_dose"] = mean_weekly_dose

            # Normalize or not
            if data_level == "normalized":
                merged["hospitalization_rate"] = normalize(merged["hospitalization_rate"])
                merged["death_rate"] = normalize(merged["death_rate"])
            else:
                merged["hospitalization_rate"] /= 100
                merged["death_rate"] /= 100

            df = merged.rename(columns={"region": "region"})
            return df[["region", "hospitalization_rate", "death_rate", "vaccine_supply_ratio", "mean_weekly_dose"]].dropna()


        except Exception as e:
            print(f"[ERROR loading FLU data]: {e}")
            return None

    else:
        print("❌ Only COVID or FLU are supported for data integration.")
        return None


In [150]:
import gym 
from gym import spaces
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import sys
import os

class BaseVaccineEnv(gym.Env):
    def __init__(self, max_steps=100, data_df=None):
        super(BaseVaccineEnv, self).__init__()
        self.max_steps = max_steps
        self.real_data = data_df
        self.state = None
        self.current_step = 0
        self.region_names = ["North", "South", "East", "West", "Central"]  # consistent for all


In [151]:
class CovidEnv(BaseVaccineEnv):
    def __init__(self, max_steps=100, data_df=None):
        super(CovidEnv, self).__init__(max_steps, data_df)
        self.state_size = 6
        self.observation_space = spaces.Box(low=0, high=1, shape=(self.state_size,), dtype=np.float32)
        self.action_space = spaces.Discrete(6)
        self.region_data = data_df  # Real data with 'region' column
        self.region_names = list(self.region_data["region"])
        self.region_states = {region: self._init_region_state(region) for region in self.region_names}

    def _init_region_state(self, region):
        row = self.region_data[self.region_data["region"] == region].iloc[0]
        return {
            "infected": float(row["hospitalization_rate"]),
            "vaccinated": float(row["vaccine_supply_ratio"]),
            "hospitalized": float(row["hospitalization_rate"])
        }

    def _get_initial_state(self):
        avg = self.region_data.mean(numeric_only=True)
        return np.array([
            avg["hospitalization_rate"],
            avg["death_rate"],
            avg["vaccine_supply_ratio"],
            0.2,  # initial coverage
            0.1,  # wastage
            avg["mean_weekly_dose"]
        ], dtype=np.float32)

    def step(self, action):
        next_state = self.state.copy()
        next_state[2] = min(1.0, next_state[2] + random.uniform(0.01, 0.03))
        next_state[3] += random.uniform(0.0, 0.01)
        reduction = next_state[2] * 0.1
        next_state[0] = max(0, next_state[0] - reduction)
        next_state[1] = max(0, next_state[1] - reduction * 0.5)
        self.state = np.clip(next_state, 0, 1).astype(np.float32)

        for region in self.region_names:
            r = self.region_states[region]
            r["vaccinated"] = min(1.0, r["vaccinated"] + random.uniform(0.01, 0.03))
            r["infected"] = max(0.0, r["infected"] - reduction * random.uniform(0.1, 0.2))
            r["hospitalized"] = max(0.0, r["hospitalized"] - reduction * random.uniform(0.05, 0.1))

        self.current_step += 1
        done = self.current_step >= self.max_steps
        return self.state, self.calculate_reward(self.state), done, {}

    def calculate_reward(self, state):
        hosp, death, coverage, waste = state[0], state[1], state[2], state[3]
        return -10 * hosp - 15 * death + 12 * coverage - 5 * waste

    def get_infected(self, region): return self.region_states[region]["infected"]
    def get_vaccinated(self, region): return self.region_states[region]["vaccinated"]
    def get_hospitalized(self, region): return self.region_states[region]["hospitalized"]


In [152]:
class FluEnv(BaseVaccineEnv):
    def __init__(self, max_steps=100, data_df=None):
        super(FluEnv, self).__init__(max_steps, data_df)
        self.state_size = 5
        self.observation_space = spaces.Box(low=0, high=1, shape=(self.state_size,), dtype=np.float32)
        self.action_space = spaces.Discrete(4)
        self.region_data = data_df
        self.region_names = list(self.region_data["region"])
        self.region_states = {region: self._init_region_state(region) for region in self.region_names}

    def _init_region_state(self, region):
        row = self.region_data[self.region_data["region"] == region].iloc[0]
        return {
            "infected": float(row["hospitalization_rate"]),
            "vaccinated": float(row["vaccine_supply_ratio"]),
            "hospitalized": float(row["hospitalization_rate"])
        }

    def _get_initial_state(self):
        avg = self.region_data.mean(numeric_only=True)
        return np.array([
            avg["hospitalization_rate"],
            avg["death_rate"],
            0.2,  # initial coverage
            avg["vaccine_supply_ratio"],
            avg["mean_weekly_dose"]
        ], dtype=np.float32)

    def step(self, action):
        next_state = self.state.copy()
        next_state[2] += random.uniform(0.01, 0.02)
        next_state[0] *= 0.95
        next_state[1] *= 0.97
        self.state = np.clip(next_state, 0, 1).astype(np.float32)

        for region in self.region_names:
            r = self.region_states[region]
            r["vaccinated"] = min(1.0, r["vaccinated"] + random.uniform(0.01, 0.02))
            r["infected"] *= random.uniform(0.9, 0.97)
            r["hospitalized"] *= random.uniform(0.93, 0.99)

        self.current_step += 1
        done = self.current_step >= self.max_steps
        return self.state, self.calculate_reward(self.state), done, {}

    def calculate_reward(self, state):
        if np.isnan(state[:3]).any():
            return -100
        hosp, death, coverage = state[0], state[1], state[2]
        return -8 * hosp - 12 * death + 10 * coverage

    def get_infected(self, region): return self.region_states[region]["infected"]
    def get_vaccinated(self, region): return self.region_states[region]["vaccinated"]
    def get_hospitalized(self, region): return self.region_states[region]["hospitalized"]


In [153]:
class MultiDiseaseEnv(BaseVaccineEnv):
    def __init__(self, disease_type="COVID", max_steps=100, data_df=None):
        assert disease_type in {"COVID", "FLU"}
        self.disease_type = disease_type
        super().__init__(max_steps=max_steps, data_df=data_df)
        self.env = CovidEnv(max_steps, data_df) if disease_type == "COVID" else FluEnv(max_steps, data_df)
        self.action_space = self.env.action_space
        self.observation_space = self.env.observation_space

    def _get_initial_state(self): return self.env._get_initial_state()
    def step(self, action): return self.env.step(action)
    def calculate_reward(self, state): return self.env.calculate_reward(state)
    def reset(self, seed=None, options=None): return self.env.reset(seed=seed, options=options)
    
    def get_infected(self, region): return self.env.get_infected(region)
    def get_vaccinated(self, region): return self.env.get_vaccinated(region)
    def get_hospitalized(self, region): return self.env.get_hospitalized(region)
    @property
    def region_names(self): return self.env.region_names


In [154]:
# ----------------------
# DQN
# ----------------------
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [155]:
import os
import sys
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Runtime Controller
# ----------------------
if __name__ == "__main__":
    print("Select disease type: [COVID / FLU]")
    disease = input("Disease type: ").strip().upper()

    print("Select data level: [raw / normalized]")
    data_level = input("Data Level: ").strip().lower()

    print("Select geographic level: [national / regional / state]")
    geo_level = input("Geo level: ").strip().lower()

    print("Optional: Enter time frame (YYYY-MM-DD). Press Enter to skip.")
    start_date = input("Start date: ").strip() or None
    end_date = input("End date: ").strip() or None

    if disease not in {"COVID", "FLU"} or data_level not in {"raw", "normalized"} or geo_level not in {"national", "regional", "state"}:
        print("❌ Invalid input. Exiting.")
        sys.exit(1)

    try:
        if start_date and end_date:
            print(f"📅 Running simulation for date range: {start_date} to {end_date}")
        elif start_date:
            print(f"📅 Running simulation starting from: {start_date}")
        elif end_date:
            print(f"📅 Running simulation up to: {end_date}")
        else:
            print("📅 Running simulation on full available data")
    except Exception as e:
        print(f"❌ Invalid date format: {e}")
        sys.exit(1)

    data = load_data(disease, data_level, geo_level, start_date, end_date)
    if data is None or data.empty:
        print("❌ Failed to load data. Exiting.")
        sys.exit(1)

    if disease == "COVID":
        env = CovidEnv(data_df=data, max_steps=50)
    elif disease == "FLU":
        env = FluEnv(data_df=data, max_steps=50)
    else:
        env = MultiDiseaseEnv(data_df=data, max_steps=50)

    state_dim = env.state_size
    action_dim = env.action_space.n
    q_network = DQN(state_dim, action_dim)
    optimizer = optim.Adam(q_network.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()

    num_episodes = 300
    gamma = 0.95
    epsilon = 1.0
    epsilon_min = 0.05
    epsilon_decay = 0.995

    reward_history = []
    loss_history = []
    simulation_frames = []

    for episode in range(num_episodes):
        state = env.reset()
        state = torch.tensor(state, dtype=torch.float32)
        total_reward = 0
        total_loss = 0
        step_count = 0
        episode_frames = []

        for t in range(50):
            action = random.choice(range(action_dim)) if random.random() < epsilon else torch.argmax(q_network(state)).item()
            next_state, reward, done, _ = env.step(action)
            next_state = torch.tensor(next_state, dtype=torch.float32)
            target = reward + gamma * torch.max(q_network(next_state))
            prediction = q_network(state)[action]
            loss = loss_fn(prediction, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # 🔹 Save per-step region states
            frame = {
                "step": t,
                "regions": [
                    {
                        "name": region,
                        "infected": env.get_infected(region),
                        "vaccinated": env.get_vaccinated(region),
                        "hospitalized": env.get_hospitalized(region)
                    }
                    for region in env.region_names
                ]
            }
            episode_frames.append(frame)

            state = next_state
            total_reward += reward
            total_loss += loss.item()
            step_count += 1
            if done:
                break

        reward_history.append(total_reward)
        loss_history.append(total_loss / step_count if step_count > 0 else 0)
        simulation_frames.append({
            "episode": episode,
            "frames": episode_frames
        })

        epsilon = max(epsilon * epsilon_decay, epsilon_min)
        print(f"Episode {episode + 1}: Total Reward = {total_reward:.2f}, Avg Loss = {loss_history[-1]:.4f}")

    # 🔻 Save region simulation frames
    vis_dir = "C:\\Users\\naing\\MultiVacSim\\visualizations"
    os.makedirs(vis_dir, exist_ok=True)
    vis_file = os.path.join(vis_dir, f"{disease.lower()}_{data_level}_{geo_level}_simulation_frames.json")

    with open(vis_file, "w") as f:
        json.dump(simulation_frames, f, indent=2)

    print(f"\n✅ Region-level simulation frames saved to:\n{vis_file}")


Select disease type: [COVID / FLU]
Select data level: [raw / normalized]
Select geographic level: [national / regional / state]
Optional: Enter time frame (YYYY-MM-DD). Press Enter to skip.
📅 Running simulation on full available data
📥 Loading files from Google Drive...


C:\Users\naing\AppData\Local\Temp\ipykernel_4452\3950639955.py:26: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(fh)


[DEBUG] hosp shape: (120003, 9)
[DEBUG] deaths shape: (194040, 14)
[DEBUG] vaccine shape: (4023, 13)
[DEBUG] hosp columns: ['State', 'Season', '_WeekendDate', 'AgeCategory_Legend', 'Sex_Label', 'Race_Label', 'Type', 'WeeklyRate', 'CumulativeRate']
[DEBUG] deaths columns: ['Data As Of', 'Start Date', 'End Date', 'Group', 'Year', 'Month', 'MMWR Week', 'Week-Ending Date', 'HHS Region', 'Race and Hispanic Origin Group', 'Age Group', 'COVID-19 Deaths', 'Total Deaths', 'Footnote']
[DEBUG] vaccine columns: ['Cumulative_Doses', 'Influenza_Season', 'Setting', 'Week_ID', 'Current_Season_Week_Ending_Label', 'MMWR_Week_Order', 'MMWR_Year', 'MMWR_Week', 'MMWR_Day', 'Dose', 'Current_Through', 'Location_and_Flu_Season_Order', 'Age_Group']
[DEBUG] Selected hospitalization column: CumulativeRate
[DEBUG] Selected death column: COVID-19 Deaths
[DEBUG] hosp rows after dropping NaNs: 120003
[DEBUG] deaths rows after dropping NaNs: 152813
[DEBUG] hosp columns: ['State', 'Season', '_WeekendDate', 'AgeCategor

SystemExit: 1

C:\Users\naing\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import json
output_dir = "C:\\Users\\naing\\MultiVacSim\\visualizations"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, f"{disease.lower()}_{data_level}_{geo_level}_simulation_frames.json")
with open(output_path, "w") as f:
    json.dump(simulation_frames, f, indent=2)

print(f"✅ Simulation frames saved to: {output_path}")
